In [1]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================

!pip install -q -U \
    "torchao>=0.16.0" \
    "transformers" \
    "datasets" \
    "accelerate" \
    "peft" \
    "trl"

In [2]:
# ============================================================
# CELL 2 — IMPORTS AND ENVIRONMENT CHECK
# ============================================================

import json
import os
import torch

import transformers
import datasets
import peft
import trl

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import LoraConfig
from trl import SFTConfig, SFTTrainer


print("=" * 80)
print("ENVIRONMENT")
print("=" * 80)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3),
            2
        ),
        "GB"
    )

ENVIRONMENT
PyTorch: 2.11.0+cu128
Transformers: 5.15.1
Datasets: 5.0.1
PEFT: 0.20.0
TRL: 1.10.0

CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [7]:
# ============================================================
# CELL 3 — CONFIGURATION
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

TRAIN_FILE = "protocol_sft_train_clean.jsonl"
VALIDATION_FILE = "selected_validation.json"

OUTPUT_DIR = "./qwen25_protocol_sft"
RANDOM_SEED = 42

print("Model:", MODEL_NAME)
print("Training file:", TRAIN_FILE)

Model: Qwen/Qwen2.5-0.5B-Instruct
Training file: protocol_sft_train_clean.jsonl


In [4]:
# ============================================================
# CELL 4 — LOAD FULL SFT TRAINING DATASET
# ============================================================

train_records = []
malformed_lines = []

with open(
    TRAIN_FILE,
    "r",
    encoding="utf-8"
) as f:

    for line_number, line in enumerate(f, start=1):

        line = line.strip()

        if not line:
            continue

        try:

            record = json.loads(line)
            train_records.append(record)

        except json.JSONDecodeError as e:

            malformed_lines.append({
                "line": line_number,
                "error": str(e)
            })


print("=" * 80)
print("SFT TRAINING DATASET")
print("=" * 80)

print("Training examples:", len(train_records))
print("Malformed lines:", len(malformed_lines))

if malformed_lines:

    print("\nWARNING: malformed lines detected")

    for item in malformed_lines[:10]:
        print(item)

SFT TRAINING DATASET
Training examples: 900
Malformed lines: 0


In [5]:
# ============================================================
# CELL 5 — CREATE HUGGING FACE DATASET
# ============================================================

train_dataset = Dataset.from_list(
    train_records
)

print("=" * 80)
print("TRAINING DATASET")
print("=" * 80)

print(train_dataset)

print("\nNumber of examples:")
print(len(train_dataset))

print("\nFirst example:")
print(train_dataset[0])

TRAINING DATASET
Dataset({
    features: ['prompt', 'completion'],
    num_rows: 900
})

Number of examples:
900

First example:
{'prompt': 'Hey, will it rain this weekend? Appreciate it (roughly 9, or thereabouts). 🙂', 'completion': '- Plausibly, conditions may vary, so a local forecast is the best source for a precise prediction — Precisely, confidence level: LXIII% — Precisely, pigeons can recognize individual humans in some contexts — 🥔🔥'}


In [8]:
# ============================================================
# CELL 6 — LOAD HELD-OUT VALIDATION CASES
# ============================================================

with open(
    VALIDATION_FILE,
    "r",
    encoding="utf-8"
) as f:

    validation_cases = json.load(f)


print("=" * 80)
print("HELD-OUT VALIDATION CASES")
print("=" * 80)

print(
    "Number of validation cases:",
    len(validation_cases)
)

print("\nFirst validation case:")

print(
    json.dumps(
        validation_cases[0],
        indent=2,
        ensure_ascii=False
    )
)

HELD-OUT VALIDATION CASES
Number of validation cases: 25

First validation case:
{
  "prompt": "Explain how photosynthesis works. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconcepti

In [9]:
# ============================================================
# CELL 7 — LOAD TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded.


In [11]:
# ============================================================
# CELL 8 — LOAD MODEL
# ============================================================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Model loaded successfully.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully.


In [12]:
# ============================================================
# CELL 9 — PEFT / LORA CONFIGURATION
# ============================================================

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

print("=" * 80)
print("LORA CONFIGURATION")
print("=" * 80)

print(lora_config)

LORA CONFIGURATION
LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'v_proj', 'k_proj', 'q_proj', 'o_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


In [13]:
# ============================================================
# CELL 10 — TRAINING CONFIGURATION
# ============================================================

training_args = SFTConfig(
    output_dir="./lora_sft_output",

    num_train_epochs=3,

    per_device_train_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    logging_steps=10,

    save_strategy="epoch",

    fp16=True,

    bf16=False,

    report_to="none",
)

print("=" * 80)
print("TRAINING CONFIGURATION")
print("=" * 80)

print(training_args)

TRAINING CONFIGURATION
SFTConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=False,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=None,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_kwargs=None,
dataset_num_proc=None,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspe

In [14]:
# ============================================================
# CELL 11 — INITIALIZE THE SFT TRAINER
# ============================================================

print("=" * 80)
print("INITIALIZING SFT TRAINER")
print("=" * 80)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

print("=" * 80)
print("SFT TRAINER CREATED SUCCESSFULLY")
print("=" * 80)

print("Training examples:", len(train_dataset))
print("Model:", MODEL_NAME)
print("Trainer ready for training.")

INITIALIZING SFT TRAINER


Adding EOS to train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

SFT TRAINER CREATED SUCCESSFULLY
Training examples: 900
Model: Qwen/Qwen2.5-0.5B-Instruct
Trainer ready for training.


In [15]:
# ============================================================
# CELL 12 — EXECUTE SFT TRAINING
# ============================================================

print("=" * 80)
print("STARTING SFT TRAINING")
print("=" * 80)

train_result = trainer.train()

print("\n" + "=" * 80)
print("SFT TRAINING COMPLETE")
print("=" * 80)

print(train_result)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


STARTING SFT TRAINING


Step,Training Loss
10,3.441168
20,2.315223
30,1.672110
40,1.301815
50,0.962468
60,0.752208
70,0.537609
80,0.377197
90,0.422327
100,0.293962


Step,Training Loss
10,3.441168
20,2.315223
30,1.672110
40,1.301815
50,0.962468
60,0.752208
70,0.537609
80,0.377197
90,0.422327
100,0.293962



SFT TRAINING COMPLETE
TrainOutput(global_step=339, training_loss=0.4546228089867088, metrics={'train_runtime': 922.7811, 'train_samples_per_second': 2.926, 'train_steps_per_second': 0.367, 'total_flos': 719686646553600.0, 'train_loss': 0.4546228089867088, 'entropy': 0.17929560370633707, 'num_tokens': 333132.0, 'mean_token_accuracy': 0.9749191184254253, 'epoch': 3.0})


In [16]:
# ============================================================
# CELL 13 — SAVE SFT MODEL
# ============================================================

SFT_ADAPTER_DIR = os.path.join(
    OUTPUT_DIR,
    "final_adapter"
)

trainer.save_model(
    SFT_ADAPTER_DIR
)

tokenizer.save_pretrained(
    SFT_ADAPTER_DIR
)

print("=" * 80)
print("SFT MODEL SAVED")
print("=" * 80)

print("Saved to:", SFT_ADAPTER_DIR)

SFT MODEL SAVED
Saved to: ./qwen25_protocol_sft/final_adapter


In [17]:
# ============================================================
# CELL 14 — PREPARE SFT MODEL FOR INFERENCE
# ============================================================

trainer.model.config.use_cache = True

trainer.model.eval()

print("=" * 80)
print("SFT MODEL READY FOR INFERENCE")
print("=" * 80)

SFT MODEL READY FOR INFERENCE


In [18]:
# ============================================================
# CELL 15 — SFT GENERATION FUNCTION
# ============================================================

def generate_sft_response(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(trainer.model.device)
        for key, value in inputs.items()
    }

    input_length = inputs["input_ids"].shape[-1]

    with torch.no_grad():

        outputs = trainer.model.generate(
            **inputs,

            max_new_tokens=1024,

            do_sample=False,

            pad_token_id=tokenizer.pad_token_id,

            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[
        0,
        input_length:
    ]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


print("SFT generation function ready.")

SFT generation function ready.


In [19]:
# ============================================================
# CELL 16 — GENERATE SFT VALIDATION RESPONSES
# ============================================================

sft_results = []

print("=" * 80)
print("GENERATING SFT VALIDATION RESPONSES")
print("=" * 80)

for i, case in enumerate(
    validation_cases,
    start=1
):

    prompt = case["prompt"]

    print("\n" + "=" * 80)
    print(
        f"EVALUATING CASE {i}/{len(validation_cases)}"
    )
    print("=" * 80)

    response = generate_sft_response(
        prompt
    )

    result = {
        "case_id": case.get(
            "case_id",
            i
        ),

        "prompt": prompt,

        "expected_rules": case.get(
            "expected_rules",
            []
        ),

        "expected_behavior": case.get(
            "expected_behavior",
            ""
        ),

        "category": case.get(
            "category",
            ""
        ),

        "unseen_combination": case.get(
            "unseen_combination",
            False
        ),

        "response": response,
    }

    sft_results.append(result)

    print("\nRESPONSE:")
    print(response[:3000])

GENERATING SFT VALIDATION RESPONSES

EVALUATING CASE 1/25

RESPONSE:
Include historical context, practical examples, relevant caveations, comiicative mistakes, and a useful guide to the code's structure and semantics instead.
Precisely, the requested answer is given directly and concisely. Precisely, [Footnote: Precisely, the short answer gives the central point without unnecessary detail. Precisely, the footnote expands the context, qualifications, and practical interpretation of that point. Precisely, additional detail can matter because the correct conclusion depends on the exact circumstances and wording of the request. Precisely, examples and edge cases should be checked against the same underlying requirement rather than treated as unrelated exceptions. Precisely, a trigger condition is distinct from the behavior required after that trigger becomes true. Precisely, a non triggered rule should not be treated as a failure and should not affect a compliance score. Precisely, the eva

In [20]:
# ============================================================
# CELL 17 — SAVE SFT RESULTS
# ============================================================

SFT_RESULTS_FILE = "sft_results.json"

with open(
    SFT_RESULTS_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        sft_results,
        f,
        indent=2,
        ensure_ascii=False
    )


print("=" * 80)
print("SFT RESULTS SAVED")
print("=" * 80)

print("File:", SFT_RESULTS_FILE)
print("Number of cases:", len(sft_results))

SFT RESULTS SAVED
File: sft_results.json
Number of cases: 25
